In [3]:
import networkx as nx
import numpy as np
from scipy.sparse import csc_matrix, diags
import pandas as pd
import numpy as np
from functions.graph import Graph, Node
from functions.functions import PowerMethod, testRank

In [4]:
# Create the nodes
node1 = Node(number=1)
node2 = Node(number=2)
node3 = Node(number=3)
node4 = Node(number=4)

# linking edges
node1.addOut([2,3,4]) 
node1.addIn([3,4]) 
node2.addIn([1]) 
node2.addOut([3,4]) 
node3.addIn([1,2,4]) 
node3.addOut([1]) 
node4.addIn([2,1]) 
node4.addOut([1,3]) 

graph = Graph([node1, node2, node3, node4])
graph.print()
graph.plot()
testRank(graph)

Node(number=1, out=[2. 3. 4.], in=[3. 4.])
Node(number=2, out=[3. 4.], in=[1.])
Node(number=3, out=[1.], in=[1. 2. 4.])
Node(number=4, out=[1. 3.], in=[2. 1.])

Generazione grafico stabile con layout circolare...


Nodi totali: 4
Nodi dangling: 0
Convergenza raggiunta all'iterazione 24 (Diff: 6.56e-09)

--- Rank ---
Rank  Score        ID     URL
------------------------------------------------------------
1     0.36815068   1     
2     0.28796163   3     
3     0.20207834   4     
4     0.14180936   2     


In [5]:
node5 = Node(number=5)
node3.addOut([5])
node3.addIn([5])
node5.addOut([3])
node5.addIn([3])
graph.addNode(node5)

testRank(graph)
graph.print()
graph.plot()

Nodi totali: 5
Nodi dangling: 0
Convergenza raggiunta all'iterazione 38 (Diff: 7.63e-09)

--- Rank ---
Rank  Score        ID     URL
------------------------------------------------------------
1     0.34889409   3     
2     0.23714058   1     
3     0.17827999   5     
4     0.13849551   4     
5     0.09718983   2     
Node(number=1, out=[2. 3. 4.], in=[3. 4.])
Node(number=2, out=[3. 4.], in=[1.])
Node(number=3, out=[1. 5.], in=[1. 2. 4. 5.])
Node(number=4, out=[1. 3.], in=[2. 1.])
Node(number=5, out=[3.], in=[3.])

Generazione grafico stabile con layout circolare...


## PREPROCESSING 
### URL Mapping and List of Edges
First of all we need to map all the url with the respective url ID. 
Then it is important to create a list of all the edges in the .dat file.

Doing so, we come up with a list of nodes and a list of edges, now - using networkx - we can create the directed graph of the web network.

We always keep track of the number of nodes and the shifted orde of nodes id (from 1 to N+1 instead of 0 to N)

In [ ]:
url_map = {}
edges_list = []
with open('hollins.dat', 'r', encoding='utf-8') as f:
    title = f.readline().split()
    nodes_number = int(title[0])
    edges_number = int(title[1])

    for i in range(nodes_number):
        line = f.readline().strip().split(' ', 1)
        node_id = int(line[0])
        url_map[node_id] = line[1]

    for line in f:
        source, destination = map(int, line.strip().split())
        edges_list.append((source, destination))

G = nx.DiGraph()
G.add_nodes_from(range(1, nodes_number + 1))
G.add_edges_from(edges_list)

ordered_nodes = list(range(1, nodes_number + 1))
N = nodes_number



- Adjacency Matrix: Constructs the sparse matrix for the graph, strictly enforcing the order defined in ordered_nodes.

- Normalization Factors: Calculates the inverse out-degree ($1/n$) for each node to weigh the links, handling dangling nodes (0 outgoing links) by setting their weight to 0.0.

- Link Matrix: Transposes the matrix (so columns represent source nodes) and applies the weights to create the column-stochastic matrix, converting binary links into transition probabilities1.

In [20]:
A_nx = nx.adjacency_matrix(G, nodelist=ordered_nodes) # Nodelist keeps track of the correct order of nodes

out_degrees = np.array([G.out_degree(n) for n in ordered_nodes], dtype=float)

with np.errstate(divide='ignore'):
    inv_out_degrees = 1.0 / out_degrees
inv_out_degrees[np.isinf(inv_out_degrees)] = 0.0

D_inv = diags(inv_out_degrees)

A_initial = A_nx.T.dot(D_inv)

is_dangling = (out_degrees == 0)

In [21]:
def PowerMethod(A, N, m=0.15, dangling_nodes=None, relTol=1e-10, maxiter=500):
    # Initialize s (vector of lenght N with all values = 1/N)
    s = np.full((N, 1), 1.0 / N)

    # Initialize and normalize xk (the starting vector)
    xk = s.copy()
    xk = xk / np.linalg.norm(xk, ord=1) 
    k = 0 
    lambda_k = 0

    while k < maxiter:
        
        dangling_sum = np.sum(xk[dangling_nodes])

        dangling_part = dangling_sum * s
        # Compute the new vector x_tilde_k+1, this is a vector NOT NORMALIZED
        # This computation exploits the sparsity of the matrix A by computing the @ product of A @ xk

        xk_new_tilde = (1 - m) * (A @ xk + dangling_part) + (m * s) 
                        
        # Rayleigh quotient - Compute the eigenvalue
        lambda_k_new = float((xk.T @ xk_new_tilde) / (xk.T @ xk))
        # Normalize the xk_tilde
        norm_val = np.linalg.norm(xk_new_tilde, ord=1)
        xk_new = xk_new_tilde / norm_val
        
        #Check the convergence
        diff = np.abs(lambda_k_new - lambda_k)
        if diff < relTol * np.abs(lambda_k_new):

            print(f"Convergenza raggiunta all'iterazione {k+1}")
            xk = xk_new
            break
        
        xk = xk_new
        k += 1
        lambda_k = lambda_k_new

    print(lambda_k)
    return xk

In [22]:
m = 0.15
pagerank_vector = PowerMethod(A_initial, N, m, dangling_nodes=is_dangling,).flatten()

ranking = []
for i in range(N):
    node_id = ordered_nodes[i]
    score = pagerank_vector[i]
    url = url_map.get(node_id, "URL non trovato")
    ranking.append((score, node_id, url))

ranking.sort(key=lambda item: item[0], reverse=True)

print(f"\n🥇 RANKING POWER METHOD (m={m})")
print("-" * 85)
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (score, node_id, url) in enumerate(ranking[:10], start=1):
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")



nx_scores_dict = nx.pagerank(G, alpha=0.85)
nx_scores = [nx_scores_dict[n] for n in sorted(G.nodes())]

ranking_nx = []
for i in range(N):
    node_id = ordered_nodes[i] 
    score = nx_scores[i]
    url = url_map.get(node_id, "URL non trovato")
    ranking_nx.append((score, node_id, url))

ranking_nx.sort(key=lambda item: item[0], reverse=True)


print(f"\n🥇 RANKING NETWORKX (m={m})")
print("-" * 85)
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (score, node_id, url) in enumerate(ranking_nx[:10], start=1):
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")

Convergenza raggiunta all'iterazione 106
0.9999999999928829

🥇 RANKING POWER METHOD (m=0.15)
-------------------------------------------------------------------------------------
Rank  Score           ID       URL
-------------------------------------------------------------------------------------
1     0.0198787506 2        http://www.hollins.edu/
2     0.0092876203 37       http://www.hollins.edu/admissions/visit/visit.htm
3     0.0086103930 38       http://www.hollins.edu/about/about_tour.htm
4     0.0080650307 61       http://www.hollins.edu/htdig/index.html
5     0.0080265649 52       http://www.hollins.edu/admissions/info-request/info-request.cfm
6     0.0071646430 43       http://www.hollins.edu/admissions/apply/apply.htm
7     0.0065827808 425      http://www.hollins.edu/academics/library/resources/web_linx.htm
8     0.0059892131 27       http://www.hollins.edu/admissions/admissions.htm
9     0.0055717361 28       http://www.hollins.edu/academics/academics.htm
10    0.00445246

/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_22420/1719758957.py:22: RuntimeWarning: divide by zero encountered in matmul
  lambda_k_new = float((xk.T @ xk_new_tilde) / (xk.T @ xk))
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_22420/1719758957.py:22: RuntimeWarning: overflow encountered in matmul
  lambda_k_new = float((xk.T @ xk_new_tilde) / (xk.T @ xk))
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_22420/1719758957.py:22: RuntimeWarning: invalid value encountered in matmul
  lambda_k_new = float((xk.T @ xk_new_tilde) / (xk.T @ xk))
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_22420/1719758957.py:22: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lambda_k_new = float((xk.T @ xk_new_tilde) / (xk.T @ xk))


In [23]:
nx_scores_dict = nx.pagerank(G, alpha=0.85)
nx_scores = [nx_scores_dict[n] for n in sorted(G.nodes())]

ranking_nx = []
for i in range(N):
    node_id = ordered_nodes[i] 
    score = nx_scores[i]
    url = url_map.get(node_id, "URL non trovato")
    ranking_nx.append((score, node_id, url))

ranking_nx.sort(key=lambda item: item[0], reverse=True)


print(f"\n🥇 RANKING NETWORKX (m={m})")
print("-" * 85)
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (score, node_id, url) in enumerate(ranking_nx[:10], start=1):
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")


🥇 RANKING NETWORKX (m=0.15)
-------------------------------------------------------------------------------------
Rank  Score           ID       URL
-------------------------------------------------------------------------------------
1     0.0202096409 2        http://www.hollins.edu/
2     0.0094646916 37       http://www.hollins.edu/admissions/visit/visit.htm
3     0.0087727259 38       http://www.hollins.edu/about/about_tour.htm
4     0.0082211756 61       http://www.hollins.edu/htdig/index.html
5     0.0081808451 52       http://www.hollins.edu/admissions/info-request/info-request.cfm
6     0.0072957239 43       http://www.hollins.edu/admissions/apply/apply.htm
7     0.0066747935 425      http://www.hollins.edu/academics/library/resources/web_linx.htm
8     0.0061110830 27       http://www.hollins.edu/admissions/admissions.htm
9     0.0056801151 28       http://www.hollins.edu/academics/academics.htm
10    0.0044699068 29       http://www.hollins.edu/grad/coedgrad.htm
